# Pipeline Summary: Enterprise Predictive Maintenance & IoT Sensor Optimization

**Stack:** PySpark · Databricks · Snowflake · MLflow · scikit-learn · Delta Lake · Unity Catalog  
**Data:** NASA CMAPSS Turbofan Engine Degradation Dataset (FD001, FD003)

---

## What Was Built

An end-to-end MLOps pipeline implementing a bronze/silver/gold medallion architecture on Databricks, ingesting raw NASA turbofan sensor data through governed feature engineering layers into model-ready Gold tables — with quality-gated writes at each layer boundary and full experiment tracking via MLflow.

---

## Bronze Layer — Schema-on-Read Ingestion

Raw CMAPSS `.txt` files (space-separated, no headers, trailing null columns from NASA file formatting) were ingested via PySpark into Unity Catalog volumes and written as Delta tables. Column names were manually assigned per the NASA readme. Lineage metadata (`subset`, `split`, `ingested_at`) was added at ingest time.

**Result:** Four Bronze Delta tables — `bronze_fd001`, `bronze_fd003`, `bronze_rul_fd001`, `bronze_rul_fd003` — totaling 75,043 rows across train and test splits.

---

## Exploratory Data Analysis — Hypothesis-Driven

**Central hypothesis:** Multivariate sensor readings contain statistically significant signal for predicting remaining useful life beyond a naive baseline.

**Finding 1 — RUL distribution is right-skewed; capping required.**  
FD001 RUL ranges from 0 to 361 cycles (mean: 107.8, std: 68.9); FD003 from 0 to 524 cycles (mean: 138.1, std: 98.8). Engines at RUL=300 are sensorially indistinguishable from engines at RUL=200 — the healthy plateau carries no degradation signal. RUL capped at **125 cycles** to focus the model on the actionable maintenance window.

**Finding 2 — 8 of 26 features are zero-variance constants.**  
Sensors 1, 5, 6, 10, 16, 18, 19 and `op_setting_3` showed standard deviations at machine epsilon in FD001's single operating condition. Dropped entirely.

**Finding 3 — Strong linear signal confirmed across remaining features.**  
Pearson correlation against capped RUL reached |r| = 0.775 (`sensor_11`). Top degradation indicators: `sensor_11` (−0.775), `sensor_4` (−0.757), `sensor_12` (+0.749), `sensor_7` (+0.733). Operational settings showed near-zero correlation (−0.006, −0.010). The strength of linear signal established Ridge regression as a legitimate baseline.

**Finding 4 — Significant multicollinearity; one severe pair.**  
`sensor_9` ↔ `sensor_14`: r = **0.963**. Additional pairs at 0.83–0.85 throughout the feature matrix. Ridge regression with L2 regularization cannot resolve near-identical features at this level — coefficient instability is expected. LSTM's gating mechanism handles multicollinearity natively. `sensor_14` was dropped; `sensor_9` retained.

**Finding 5 — Class imbalance in the alert label.**  
Defining a maintenance alert as "failure within 30 cycles" yields 10.2% positive cases in FD001 and 8.1% in FD003. A naive classifier predicting "no alert" achieves 90% accuracy while catching zero failures. Recall — not accuracy — is the primary metric for the alert layer.

**Finding 6 — Temporal bias requires sequence-aware modeling.**  
Engines begin with unknown initial wear states. A row-independent model cannot distinguish an engine at RUL=50 with heavy wear history from one that started near-new. Addressed via 30-cycle rolling window features for Ridge and native sequence modeling for LSTM.

---

## Silver Layer — Feature Engineering with Quality-Gated Writes

All transformations driven by EDA findings:

1. **RUL computation and capping** — derived from max cycle per engine; capped at 125
2. **Zero-variance removal** — 8 features dropped
3. **Collinearity resolution** — `sensor_14` dropped (r = 0.963 with `sensor_9`)
4. **Rolling window features** — 30-cycle rolling mean and std per engine per sensor
5. **Min-max normalization** — fit on training set only to prevent data leakage; test values clipped to [0, 1]
6. **Binary alert label** — `alert_30` = 1 if RUL_capped ≤ 30, else 0

**Governance:** Data quality checks asserted null constraints, RUL bounds [0, 125], normalization bounds [0, 1], and binary label validity before each write. The first write attempt was correctly blocked when test-set sensor values fell outside the training normalization range — a real leakage detection. After clipping, all checks passed.

**Result:** `silver_fd001` — 33,727 rows, 63 columns, alert rate 10.2%  
**Result:** `silver_fd003` — 41,316 rows, 63 columns, alert rate 8.1%

---

## Pipeline Architecture
Raw .txt files → Unity Catalog Volume (landing zone)
↓
Bronze Delta tables — schema-on-read, lineage metadata
↓ [quality gate]
Silver Delta tables — 63 features, RUL cap, rolling windows, normalized
↓ [MLflow tracking]
Gold layer — model predictions, RUL scores, alert classifications
↓
FastAPI endpoint → Streamlit business dashboard

In [0]:
# ============================================================
# CELL 1 — Silver Layer: Feature Engineering
# Notebook: 03_silver_layer
# Transforms: drop zero-variance, cap RUL, rolling windows,
#             normalize, resolve collinearity, write Silver Delta
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# --- Load Bronze tables (train only for feature engineering) ---
bronze_fd001 = spark.table("workspace.predictive_maintenance.bronze_fd001")
bronze_fd003 = spark.table("workspace.predictive_maintenance.bronze_fd003")

# --- EDA-driven constants ---
RUL_CAP = 125

FEATURES_TO_DROP = [
    "sensor_1", "sensor_5", "sensor_6", "sensor_10",
    "sensor_16", "sensor_18", "sensor_19", "op_setting_3"
]

# Drop sensor_14 to resolve 0.963 collinearity with sensor_9
# sensor_9 retained — slightly more variance across FD003
COLLINEAR_DROP = ["sensor_14"]

ROLLING_WINDOW = 30  # cycles — gives Ridge temporal context

INFORMATIVE_SENSORS = [
    "sensor_2", "sensor_3", "sensor_4", "sensor_7", "sensor_8",
    "sensor_9", "sensor_11", "sensor_12", "sensor_13",
    "sensor_15", "sensor_17", "sensor_20", "sensor_21"
]

def build_silver(df, subset_label):
    """
    Apply all EDA-driven transformations to produce Silver features.
    Steps:
      1. Compute RUL from max cycle per engine
      2. Cap RUL at 125
      3. Drop zero-variance and collinear features
      4. Compute rolling mean + std per sensor (window=30) for temporal context
      5. Min-max normalize sensor values per subset
      6. Add metadata
    """
    # Step 1 & 2: Compute and cap RUL
    w_engine = Window.partitionBy("unit_id")
    df = (df
          .withColumn("max_cycle", F.max("time_cycles").over(w_engine))
          .withColumn("RUL", F.col("max_cycle") - F.col("time_cycles"))
          .withColumn("RUL_capped", F.least(F.col("RUL"), F.lit(RUL_CAP)))
          .drop("max_cycle")
    )

    # Step 3: Drop zero-variance and collinear features
    drop_cols = FEATURES_TO_DROP + COLLINEAR_DROP
    df = df.drop(*drop_cols)

    # Step 4: Rolling mean and std per engine per sensor
    w_roll = (Window.partitionBy("unit_id")
              .orderBy("time_cycles")
              .rowsBetween(-ROLLING_WINDOW + 1, 0))

    for s in INFORMATIVE_SENSORS:
        df = (df
              .withColumn(f"{s}_roll_mean", F.mean(s).over(w_roll))
              .withColumn(f"{s}_roll_std",
                          F.stddev(s).over(w_roll))
        )

    # Fill nulls in roll_std for first window rows (< 30 cycles)
    roll_std_cols = [f"{s}_roll_std" for s in INFORMATIVE_SENSORS]
    df = df.fillna(0.0, subset=roll_std_cols)

    # Step 5: Min-max normalize original sensor columns
    # Compute min/max from training split only to avoid leakage
    train_df = df.filter(F.col("split") == "train")
    for s in INFORMATIVE_SENSORS:
        min_val = train_df.agg(F.min(s)).collect()[0][0]
        max_val = train_df.agg(F.max(s)).collect()[0][0]
        rng = max_val - min_val if max_val != min_val else 1.0
        df = df.withColumn(
            f"{s}_norm",
            (F.col(s) - F.lit(min_val)) / F.lit(rng)
        )

    # Step 6: Maintenance alert label (binary — failure within 30 cycles)
    df = df.withColumn(
        "alert_30",
        F.when(F.col("RUL_capped") <= 30, 1).otherwise(0)
    )

    return df.withColumn("silver_subset", F.lit(subset_label))

print("Building Silver FD001...")
silver_fd001 = build_silver(bronze_fd001, "FD001")

print("Building Silver FD003...")
silver_fd003 = build_silver(bronze_fd003, "FD003")

print("Done. Sample schema:")
silver_fd001.printSchema()

Building Silver FD001...
Building Silver FD003...
Done. Sample schema:
root
 |-- unit_id: integer (nullable = true)
 |-- time_cycles: integer (nullable = true)
 |-- op_setting_1: double (nullable = true)
 |-- op_setting_2: double (nullable = true)
 |-- sensor_2: double (nullable = true)
 |-- sensor_3: double (nullable = true)
 |-- sensor_4: double (nullable = true)
 |-- sensor_7: double (nullable = true)
 |-- sensor_8: double (nullable = true)
 |-- sensor_9: double (nullable = true)
 |-- sensor_11: double (nullable = true)
 |-- sensor_12: double (nullable = true)
 |-- sensor_13: double (nullable = true)
 |-- sensor_15: double (nullable = true)
 |-- sensor_17: integer (nullable = true)
 |-- sensor_20: double (nullable = true)
 |-- sensor_21: double (nullable = true)
 |-- subset: string (nullable = true)
 |-- split: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- RUL: integer (nullable = true)
 |-- RUL_capped: integer (nullable = false)
 |-- sensor_2_roll_mean

In [0]:
# ============================================================
# CELL 2 (FIXED) — Write Silver Delta Tables + Quality Checks
# Fix: clip normalized values to [0,1] after min-max scaling
# This handles test set observations outside training range
# ============================================================

from pyspark.sql import functions as F

# Cast sensor_17 to double
silver_fd001 = silver_fd001.withColumn("sensor_17", F.col("sensor_17").cast("double"))
silver_fd003 = silver_fd003.withColumn("sensor_17", F.col("sensor_17").cast("double"))

# Clip all _norm columns to [0, 1] — handles test set range violations
norm_cols = [c for c in silver_fd001.columns if c.endswith("_norm")]
for col in norm_cols:
    silver_fd001 = silver_fd001.withColumn(col, F.least(F.lit(1.0), F.greatest(F.lit(0.0), F.col(col))))
    silver_fd003 = silver_fd003.withColumn(col, F.least(F.lit(1.0), F.greatest(F.lit(0.0), F.col(col))))

print(f"Clipped {len(norm_cols)} normalized columns to [0, 1]")

# ============================================================
# DATA QUALITY CHECKS
# ============================================================

def run_quality_checks(df, name):
    errors = []

    key_cols = ["unit_id", "time_cycles", "RUL_capped", "sensor_11", "sensor_4", "sensor_12"]
    for col in key_cols:
        null_count = df.filter(F.col(col).isNull()).count()
        if null_count > 0:
            errors.append(f"NULL CHECK FAILED: {col} has {null_count} nulls")

    rul_violations = df.filter((F.col("RUL_capped") < 0) | (F.col("RUL_capped") > 125)).count()
    if rul_violations > 0:
        errors.append(f"RUL BOUNDS FAILED: {rul_violations} rows outside [0, 125]")

    uid_violations = df.filter(F.col("unit_id") <= 0).count()
    if uid_violations > 0:
        errors.append(f"UNIT_ID CHECK FAILED: {uid_violations} non-positive unit_ids")

    for col in norm_cols:
        violations = df.filter((F.col(col) < -0.01) | (F.col(col) > 1.01)).count()
        if violations > 0:
            errors.append(f"NORM BOUNDS FAILED: {col} has {violations} values outside [0,1]")

    alert_vals = df.select("alert_30").distinct().count()
    if alert_vals > 2:
        errors.append(f"ALERT LABEL FAILED: alert_30 has {alert_vals} distinct values (expected 2)")

    print(f"\n=== DATA QUALITY REPORT: {name} ===")
    if errors:
        for e in errors:
            print(f"  ❌ {e}")
        raise ValueError(f"Data quality checks failed for {name}. Blocking Silver write.")
    else:
        row_count = df.count()
        print(f"  ✅ All checks passed | {row_count:,} rows | {len(df.columns)} columns")

run_quality_checks(silver_fd001, "silver_fd001")
run_quality_checks(silver_fd003, "silver_fd003")

# ============================================================
# WRITE SILVER DELTA TABLES
# ============================================================

(silver_fd001.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.predictive_maintenance.silver_fd001"))

(silver_fd003.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.predictive_maintenance.silver_fd003"))

print("\n✅ Silver Delta tables written:")
print("   - workspace.predictive_maintenance.silver_fd001")
print("   - workspace.predictive_maintenance.silver_fd003")

for name, tbl in [("silver_fd001", "workspace.predictive_maintenance.silver_fd001"),
                   ("silver_fd003", "workspace.predictive_maintenance.silver_fd003")]:
    df = spark.table(tbl)
    alert_pct = df.filter(F.col("alert_30") == 1).count() / df.count() * 100
    print(f"\n{name}: {df.count():,} rows | alert_30=1: {alert_pct:.1f}%")

Clipped 13 normalized columns to [0, 1]

=== DATA QUALITY REPORT: silver_fd001 ===
  ✅ All checks passed | 33,727 rows | 63 columns

=== DATA QUALITY REPORT: silver_fd003 ===
  ✅ All checks passed | 41,316 rows | 63 columns

✅ Silver Delta tables written:
   - workspace.predictive_maintenance.silver_fd001
   - workspace.predictive_maintenance.silver_fd003

silver_fd001: 33,727 rows | alert_30=1: 10.2%

silver_fd003: 41,316 rows | alert_30=1: 8.1%
